In [1]:
import os
import gc
import glob
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import time

In [ ]:
# ===== 設定 =====
ROOT = r"\\mis3c076\FTP_Root\WB_RPM_Log\process_data"
TOP_ONLY = False          # True = 只檢查第一層子資料夾；False = 檢查所有層
IGNORE_FILES = {"thumbs.db", "desktop.ini", ".ds_store"}   # 這些系統檔不算「有檔案」
# ================


def scan_empty_folders(root, top_only=False, ignore_files=IGNORE_FILES):
    errors = []
    no_files_in_subtree = {}
    rows = []

    def on_error(err):
        errors.append({"path": getattr(err, "filename", ""), "error": str(err)})

    # topdown=False：先處理子資料夾，再處理父資料夾
    for dirpath, dirnames, filenames in os.walk(root, topdown=False, onerror=on_error):
        real_files = [f for f in filenames if f.lower() not in ignore_files]
        child_paths = [os.path.join(dirpath, d) for d in dirnames]

        totally_empty = (len(real_files) == 0) and (len(dirnames) == 0)
        subtree_no_files = (len(real_files) == 0) and all(
            no_files_in_subtree.get(c, False) for c in child_paths
        )
        no_files_in_subtree[dirpath] = subtree_no_files

        if os.path.normpath(dirpath) != os.path.normpath(root):
            rows.append({
                "path": dirpath,
                "完全空": totally_empty,
                "子樹沒有檔案": subtree_no_files,
            })

    df = pd.DataFrame(rows, columns=["path", "完全空", "子樹沒有檔案"])

    if top_only and not df.empty:
        top_level = {
            os.path.join(root, e.name)
            for e in os.scandir(root)
            if e.is_dir(follow_symlinks=False)
        }
        df = df[df["path"].isin(top_level)]

    return df.sort_values("path").reset_index(drop=True), pd.DataFrame(errors)


if not os.path.isdir(ROOT):
    print(f"找不到資料夾或沒有權限：{ROOT}")
else:
    t0 = time.time()
    df_folders, df_errors = scan_empty_folders(ROOT, top_only=TOP_ONLY)

    print(f"檢查範圍：{'第一層子資料夾' if TOP_ONLY else '所有子資料夾（含各層）'}")
    print(f"資料夾總數：{len(df_folders)}")
    print(f"完全空的資料夾：{df_folders['完全空'].sum()}")
    print(f"整棵子樹沒有任何檔案的資料夾：{df_folders['子樹沒有檔案'].sum()}（包含上一項）")
    print(f"無法讀取的資料夾：{len(df_errors)}")
    print(f"耗時：{time.time() - t0:.1f} 秒")

    df_empty = df_folders[df_folders["子樹沒有檔案"]]
    # display(df_empty)

檢查範圍：所有子資料夾（含各層）
資料夾總數：782
完全空的資料夾：687
整棵子樹沒有任何檔案的資料夾：687（包含上一項）
無法讀取的資料夾：0
耗時：3.5 秒


In [27]:
# 只取第一層（機台）資料夾，並排除「整棵子樹沒有任何檔案」的
is_top_level = df_folders["path"].apply(
    lambda p: os.path.normpath(os.path.dirname(p)) == os.path.normpath(ROOT)
)
df_nonempty = df_folders[is_top_level & ~df_folders["子樹沒有檔案"]]

nonempty_eqp = sorted(df_nonempty["path"].apply(os.path.basename).tolist())

# print(f"有資料的機台數：{len(nonempty_eqp)}")
print(nonempty_eqp)
# print(','.join(nonempty_eqp))

['WBR0001', 'WBR0004', 'WBR0005', 'WBR0024', 'WBR0026', 'WBR0033', 'WBR0034', 'WBR0038', 'WBR1001', 'WBR1004', 'WBR1006', 'WBR1008', 'WBR1014', 'WBR1018', 'WBR1020', 'WBR1021', 'WBR1023', 'WBR1026', 'WBR1034', 'WBR1037', 'WBR1039', 'WBR1040', 'WBR1041', 'WBR1043', 'WBR1045', 'WBR1050', 'WBR2022', 'WBR2025', 'WBR2053', 'WBR2057', 'WBR2065', 'WBR2076', 'WBR2077', 'WBR2115', 'WBR2130', 'WBR2132', 'WBR2150', 'WBR2170', 'WBR2173', 'WBR2174', 'WBR2180', 'WBR2183', 'WBR2192', 'WBR2194', 'WBR2195', 'WBR2196', 'WBR2212', 'WBR2214', 'WBR2215', 'WBR2217', 'WBR2221', 'WBR2222', 'WBR2226', 'WBR2228', 'WBR2233', 'WBR2235', 'WBR2236', 'WBR2239', 'WBR2274', 'WBR4018', 'WBR4019', 'WBR4020', 'WBR6006', 'WBR6051', 'WBR6054', 'WBR6072', 'WBR6079', 'WBR6092', 'WBR6096', 'WBR6128', 'WBR6136', 'WBR6137', 'WBR6139', 'WBR6146', 'WBR6147', 'WBR6149', 'WBR6150', 'WBR6176', 'WBR6191', 'WBR6201', 'WBR6204', 'WBR6207', 'WBR6209', 'WBR6210', 'WBR6211', 'WBR6213', 'WBR6217', 'WBR6219', 'WBR6220', 'WBR6223', 'WBR6239'

In [28]:
print(','.join(nonempty_eqp))

WBR0001,WBR0004,WBR0005,WBR0024,WBR0026,WBR0033,WBR0034,WBR0038,WBR1001,WBR1004,WBR1006,WBR1008,WBR1014,WBR1018,WBR1020,WBR1021,WBR1023,WBR1026,WBR1034,WBR1037,WBR1039,WBR1040,WBR1041,WBR1043,WBR1045,WBR1050,WBR2022,WBR2025,WBR2053,WBR2057,WBR2065,WBR2076,WBR2077,WBR2115,WBR2130,WBR2132,WBR2150,WBR2170,WBR2173,WBR2174,WBR2180,WBR2183,WBR2192,WBR2194,WBR2195,WBR2196,WBR2212,WBR2214,WBR2215,WBR2217,WBR2221,WBR2222,WBR2226,WBR2228,WBR2233,WBR2235,WBR2236,WBR2239,WBR2274,WBR4018,WBR4019,WBR4020,WBR6006,WBR6051,WBR6054,WBR6072,WBR6079,WBR6092,WBR6096,WBR6128,WBR6136,WBR6137,WBR6139,WBR6146,WBR6147,WBR6149,WBR6150,WBR6176,WBR6191,WBR6201,WBR6204,WBR6207,WBR6209,WBR6210,WBR6211,WBR6213,WBR6217,WBR6219,WBR6220,WBR6223,WBR6239,WBR6240,WBR6265,WBR6267,WBR6269


### 讀取 PDT-UNEXPECTED-NSOP 資料

In [2]:
import pandas as pd

path = r"C:\Users\A134549\Downloads\event_nsop_record_1102_1108.csv"

# 依序嘗試常見編碼
for enc in ["utf-8-sig", "cp950", "latin-1"]:
    try:
        df_nsop = pd.read_csv(path, encoding=enc, low_memory=False)
        break
    except UnicodeDecodeError:
        continue

df_nsop.columns = df_nsop.columns.str.strip()   # 去掉欄位名稱前後的空白

machine_list = sorted(
    df_nsop["MACHINECODE"].dropna().astype(str).str.strip().unique().tolist()
)

print(f"資料筆數：{len(df_nsop):,}")
print(f"MACHINECODE 唯一值數量：{len(machine_list)}")
print(", ".join(machine_list))
before = len(df_nsop)
machine_list=machine_list[:20]
keep = {m.strip().upper() for m in machine_list}


df_nsop = df_nsop[
    df_nsop["MACHINECODE"].astype(str).str.strip().str.upper().isin(keep)
].reset_index(drop=True)

print(f"篩選前：{before} 筆，篩選後：{len(df_nsop)} 筆")
print(f"篩選後的機台數：{df_nsop['MACHINECODE'].nunique()}")

資料筆數：641
MACHINECODE 唯一值數量：87
WBR0001, WBR0004, WBR0005, WBR0033, WBR0034, WBR0038, WBR1001, WBR1004, WBR1006, WBR1008, WBR1014, WBR1018, WBR1020, WBR1021, WBR1023, WBR1026, WBR1034, WBR1037, WBR1039, WBR1040, WBR1041, WBR1043, WBR1050, WBR2022, WBR2025, WBR2053, WBR2057, WBR2065, WBR2076, WBR2077, WBR2115, WBR2130, WBR2132, WBR2150, WBR2170, WBR2173, WBR2174, WBR2180, WBR2183, WBR2192, WBR2194, WBR2195, WBR2212, WBR2214, WBR2215, WBR2217, WBR2221, WBR2222, WBR2226, WBR2228, WBR2233, WBR2235, WBR2236, WBR2239, WBR2274, WBR4018, WBR4019, WBR4020, WBR6006, WBR6072, WBR6079, WBR6092, WBR6096, WBR6128, WBR6136, WBR6137, WBR6139, WBR6146, WBR6147, WBR6149, WBR6150, WBR6176, WBR6191, WBR6201, WBR6204, WBR6207, WBR6209, WBR6210, WBR6211, WBR6213, WBR6217, WBR6219, WBR6220, WBR6223, WBR6239, WBR6240, WBR6267
篩選前：641 筆，篩選後：186 筆
篩選後的機台數：20


In [3]:
df_nsop.head()

,SHIFTDATE,CLASS,DNSHIFT,PROCESS,LOCATION,FLOOR,MACHINECODE,LOTNO,PROGRAMNAME,EVENTCODE,EVENTMESSAGE,EVENTTIME,MAJORCODEID,MUBA,MFGDEPT,DEVICE,INSERTDT,DATA_YEAR
0,2026-11-06 00:00:00,5,N,W/B,3A,5F,WBR1014,2610T2635.06,SMIM32DGG308-01-EM-01,142,PDT-UNEXPECTED-NSOP,2026/11/06 19:58:36,0,1,AP 33,NC14N17M51PTP34352-STK-N47-HVM(CTRL,2026/11/06 20:30:48,2026
1,2026-11-06 00:00:00,5,N,W/B,3A,5F,WBR1034,M26BZ0592.00,SMIM97AGG328-00-MP-00,142,PDT-UNEXPECTED-NSOP,2026/11/06 19:55:44,0,1,AP 33,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,2026/11/06 20:30:48,2026
2,2026-11-06 00:00:00,5,N,W/B,3A,5F,WBR1034,M26BZ0592.00,SMIM97AGG328-00-MP-00,142,PDT-UNEXPECTED-NSOP,2026/11/06 20:59:41,0,1,AP 33,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,2026/11/06 21:30:48,2026
3,2026-11-06 00:00:00,5,N,W/B,2B,4F,WBR0033,M26BZ0388.00,SMIM63HDG404-00-AM-02,142,PDT-UNEXPECTED-NSOP,2026/11/06 21:36:51,0,1,AP 23,Y62P6JR96PTP31364-MSH-HVM2-CIP(M2,2026/11/06 22:30:47,2026
4,2026-11-06 00:00:00,5,N,W/B,2B,4F,WBR1006,M26BY9546.00,SMIM54EGG303-01-KF-01,142,PDT-UNEXPECTED-NSOP,2026/11/06 22:20:12,0,1,AP 23,NC17P17D6PTP36045-K5A-SMBP-HVM(M1,2026/11/06 22:30:49,2026


In [32]:
import os
import io
import re
import gc
import time
import tarfile
import warnings
import pandas as pd

warnings.filterwarnings("ignore", message="The behavior of DataFrame concatenation with empty or all-NA entries")



# ===== 設定 =====
ROOT = r"\\mis3c076\FTP_Root\WB_RPM_Log\process_data"
OUT_DIR = r"C:\Users\A134549\rpm_parquet_11021108"     # 存到本機硬碟，不要存回網路磁碟
DATE_START = "2026-11-02"                      # 只保留這段期間的資料（含）
DATE_END = "2026-11-08"                        # 不含這一天；設成 None 則不篩選日期
KEEP_COLS = None                               # None = 全部欄位；或填欄位名稱 list 只留需要的
MAX_FILES_PER_EQP = None                       # 測試時先設成 1
ONLY_DEEPEST = True                            # True = 每個壓縮檔只取最深一層的 csv
ENCODINGS = ["utf-8-sig", "cp950", "latin-1"]  # 依序嘗試的編碼
# ================

os.makedirs(OUT_DIR, exist_ok=True)

# 檔名格式：..._2026-11-01T02-25-22_2026-11-01T03-25-22.csv（開始時間_結束時間）
FNAME_TIME = re.compile(
    r"(\d{4}-\d{2}-\d{2})T(\d{2})-(\d{2})-(\d{2})_(\d{4}-\d{2}-\d{2})T(\d{2})-(\d{2})-(\d{2})"
)
skipped_csv = 0


def csv_in_range(filename):
    """依檔名的開始、結束時間判斷要不要讀。檔名格式不符時一律讀取。"""
    m = FNAME_TIME.search(filename)
    if not m:
        return True
    start = pd.Timestamp(f"{m.group(1)} {m.group(2)}:{m.group(3)}:{m.group(4)}")
    end = pd.Timestamp(f"{m.group(5)} {m.group(6)}:{m.group(7)}:{m.group(8)}")
    if DATE_END and start >= pd.Timestamp(DATE_END):    # 開始和結束都在範圍之後
        return False
    if DATE_START and end < pd.Timestamp(DATE_START):   # 整段都在範圍之前
        return False
    return True


def read_csv_bytes(raw):
    """用多種編碼嘗試讀取 csv 內容。"""
    last_err = None
    for enc in ENCODINGS:
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc, low_memory=False)
        except UnicodeDecodeError as e:
            last_err = e
    raise last_err


def read_tarxz(path):
    """讀一個 .tar.xz，回傳 (合併後的 DataFrame 或 None, 錯誤清單)。"""
    global skipped_csv
    found, errs = [], []
    # 'r|xz' 是串流模式：只解壓一次，網路磁碟上比隨機讀取快很多
    with tarfile.open(path, mode="r|xz") as tar:
        for m in tar:
            name = m.name
            base = os.path.basename(name)
            if not m.isfile() or not base.lower().endswith(".csv") or base.startswith("._"):
                continue
            if not csv_in_range(base):          # 依檔名時間跳過不需要的 csv
                skipped_csv += 1
                continue
            try:
                raw = tar.extractfile(m).read()
                if not raw.strip():
                    continue
                df = read_csv_bytes(raw)
                if df.empty:
                    continue
                found.append((name.strip("/").count("/"), name, df))
            except Exception as e:
                errs.append({"archive": path, "member": name, "error": str(e)})
    if not found:
        return None, errs
    if ONLY_DEEPEST:
        max_depth = max(d for d, _, _ in found)
        found = [x for x in found if x[0] == max_depth]
    parts = []
    for _, name, df in found:
        df["source_csv"] = name
        parts.append(df)
    return pd.concat(parts, ignore_index=True), errs


def shrink(df):
    """數值欄位降精度，節省記憶體與硬碟空間。"""
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].astype("float32")
    for c in df.select_dtypes("int64").columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


# 只處理 machine_list 裡、在 ROOT 底下有資料夾的機台
target_eqp = sorted(m.strip() for m in machine_list if os.path.isdir(os.path.join(ROOT, m.strip())))
print(f"machine_list：{len(machine_list)} 台，找得到資料夾：{len(target_eqp)} 台\n")

error_log = []
written, skipped, total_rows = 0, 0, 0
t0 = time.time()

for eqp in target_eqp:
    tar_files = []
    for dirpath, _, filenames in os.walk(os.path.join(ROOT, eqp)):
        tar_files += [os.path.join(dirpath, f) for f in filenames if f.lower().endswith(".tar.xz")]
    tar_files.sort()
    if MAX_FILES_PER_EQP:
        tar_files = tar_files[:MAX_FILES_PER_EQP]
    print(f"{eqp}：{len(tar_files)} 個 .tar.xz")

    for path in tar_files:
        out_name = f"{eqp}__{os.path.basename(path).replace('.tar.xz', '')}.parquet"
        out_path = os.path.join(OUT_DIR, out_name)
        if os.path.exists(out_path):          # 已經處理過就跳過，中斷後可以接著跑
            skipped += 1
            continue
        df = None
        try:
            df, errs = read_tarxz(path)
            error_log.extend(errs)
            if df is None:
                continue

            if KEEP_COLS:
                df = df[[c for c in KEEP_COLS if c in df.columns] + ["source_csv"]].copy()

            if DATE_START and "Timestamp" in df.columns:
                ts = pd.to_datetime(df["Timestamp"], errors="coerce")
                mask = ts >= pd.Timestamp(DATE_START)
                if DATE_END:
                    mask &= ts < pd.Timestamp(DATE_END)
                df = df[mask].copy()
                if df.empty:
                    continue

            df["source_eqp"] = eqp
            df["source_archive"] = os.path.basename(path)
            df = shrink(df)
            df.to_parquet(out_path, index=False)
            written += 1
            total_rows += len(df)
        except Exception as e:
            error_log.append({"archive": path, "member": "", "error": str(e)})
        finally:
            df = None
            gc.collect()

df_errors_tar = pd.DataFrame(error_log)
print(f"\n新寫入 parquet 檔：{written}，已存在跳過：{skipped}")
print(f"新寫入資料筆數：{total_rows:,}")
print(f"依檔名時間跳過的 csv：{skipped_csv}")
print(f"讀取失敗：{len(df_errors_tar)} 筆")
print(f"耗時：{time.time() - t0:.1f} 秒")

machine_list：20 台，找得到資料夾：20 台

WBR0001：16 個 .tar.xz
WBR0004：18 個 .tar.xz
WBR0005：18 個 .tar.xz
WBR0033：1 個 .tar.xz
WBR0034：1 個 .tar.xz
WBR0038：1 個 .tar.xz
WBR1001：18 個 .tar.xz
WBR1004：17 個 .tar.xz
WBR1006：1 個 .tar.xz
WBR1008：13 個 .tar.xz
WBR1014：16 個 .tar.xz
WBR1018：17 個 .tar.xz
WBR1020：17 個 .tar.xz
WBR1021：18 個 .tar.xz
WBR1023：10 個 .tar.xz
WBR1026：1 個 .tar.xz
WBR1034：18 個 .tar.xz
WBR1037：15 個 .tar.xz
WBR1039：14 個 .tar.xz
WBR1040：18 個 .tar.xz

新寫入 parquet 檔：152，已存在跳過：0
新寫入資料筆數：26,840,740
依檔名時間跳過的 csv：865
讀取失敗：0 筆
耗時：432.2 秒


WBR0034因為CSV裡面的Timestamp超出日期範圍所以沒有下載

### 讀取 W/B LOG (Parquet)

In [4]:
def read_dir(path, cols=None, max_files=None):
    parts = []
    files = sorted(glob.glob(os.path.join(path, "*.parquet")))[:max_files]
    for f in files:
        names = pq.read_schema(f).names
        d = pd.read_parquet(f, columns=[c for c in cols if c in names] if cols else None)
        for c in d.columns:
            if pd.api.types.is_numeric_dtype(d[c]) and not pd.api.types.is_bool_dtype(d[c]):
                d[c] = d[c].astype("float32")
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

In [5]:
cols = ['Timestamp',	'EQP',	'Recipe',	'Device',	'Bond Program',	'Lot Id',	'Output Mag Id',	'Output Mag Slot',	'Leadframe Id',	'Index Id'	,'Device Id',	'Device Column','Device Row',	'Ref Sys',	'EP Set',	'EP 1 Score',	'EP 2 Score',	'Wire Id',	'Bond Seq Num'	,'Bond Type',	'RPM PRS ID',	'Die Height (mil)',	'Die Tilt (deg)','Die Placement X (mil)',
 'Die Placement Y (mil)','Die Rotation (degfor)',	'Bond Height Delta (mil)'	,'Deformation (Cont) (um)'	,'Deformation (TD) (um)','Ball Diam (mil)','Ball Place X (mil)','Ball Place Y (mil)','USG current (mA)'	,'Force','EFO Voltage Avg (V)'	,'Z at Contact (um)',	'Z at End of Bonding (um)'	,'Bond Height Detection (um)','Ball Height (TD) (um)','Ball Height (Cont) (um)'	,'Impedance Ratio 1','Impedance Ratio 2']
# df = pd.read_parquet(r"C:\Users\A134549\rpm_parquet_11021108", columns=cols)
df = read_dir(r"C:\Users\A134549\rpm_parquet_11021108", cols, max_files=3)
print(f"{len(df):,} 筆，約 {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(df.shape)
df.head()

934,409 筆，約 0.73 GB
(934409, 42)


,Timestamp,EQP,Recipe,Device,Bond Program,Lot Id,Output Mag Id,Output Mag Slot,Leadframe Id,Index Id,...,USG current (mA),Force,EFO Voltage Avg (V),Z at Contact (um),Z at End of Bonding (um),Bond Height Detection (um),Ball Height (TD) (um),Ball Height (Cont) (um),Impedance Ratio 1,Impedance Ratio 2
0,2026-11-02 05:00:00.450,WBR0001,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,UM237_3A,M26AS7585.00,12687_2026-11-01_03-54-o009,10.0,BXBM67026409-0405,1.0,...,47.939999,13.16,0.0,109.110001,107.360001,NaN,NaN,NaN,1.19,1.08
1,2026-11-02 04:59:59.983,WBR0001,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,UM237_3A,M26AS7585.00,12687_2026-11-01_03-54-o009,10.0,BXBM67026409-0405,1.0,...,47.919998,13.02,0.0,110.589996,108.629997,0.98,NaN,NaN,1.19,1.08
2,2026-11-02 04:59:59.833,WBR0001,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,UM237_3A,M26AS7585.00,12687_2026-11-01_03-54-o009,10.0,BXBM67026409-0405,1.0,...,47.930000,13.05,0.0,109.620003,107.820000,NaN,NaN,NaN,1.19,1.08
3,2026-11-02 04:59:47.033,WBR0001,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,UM237_3A,M26AS7585.00,12687_2026-11-01_03-54-o009,10.0,BXBM67026409-0405,1.0,...,47.930000,13.43,0.0,109.199997,107.580002,-0.38,NaN,NaN,1.19,1.09
4,2026-11-02 05:00:00.300,WBR0001,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,UM237_3A,M26AS7585.00,12687_2026-11-01_03-54-o009,10.0,BXBM67026409-0405,1.0,...,47.939999,13.28,0.0,109.059998,107.330002,-0.25,NaN,NaN,1.19,1.08


In [6]:
del df
gc.collect()

0

In [10]:
# ===== 設定 =====
NSOP_PATH = r"C:\Users\A134549\Downloads\event_nsop_record_1102_1108.csv"
PARQUET_DIR = r"C:\Users\A134549\rpm_parquet_11021108"        # 之前存好的 RPM parquet
LABEL_DIR = r"C:\Users\A134549\rpm_parquet_labeled_11021108"  # 加上標記後的輸出位置
NSOP_EQP_COL = "MACHINECODE"
NSOP_TIME_COL = "EVENTTIME"     # 先跑一次看欄位名稱，再填入 NSOP 發生時間的欄位，例如 "EVENTTIME"
RPM_TIME_COL = "Timestamp"
MATCH_TOL_MIN = 5        # NSOP 時間和最近一筆 RPM 紀錄最多差幾分鐘，仍視為對得上
# ================


In [11]:
# assert NSOP_TIME_COL, "請先在設定區填入 NSOP_TIME_COL，再執行這個 cell"

# ---------- NSOP 事件整理 ----------
ctx_cols = [c for c in ["LOTNO", "PROGRAMNAME", "DEVICE"] if c in df_nsop.columns]
ev = df_nsop[[NSOP_EQP_COL, NSOP_TIME_COL] + ctx_cols].copy()#從df_nsop取出這三個欄位"LOTNO", "PROGRAMNAME", "DEVICE"，複製成新表ev。
ev["eqp"] = ev[NSOP_EQP_COL].astype(str).str.strip().str.upper()#機台編號去掉前後空白、轉大寫
raw = ev[NSOP_TIME_COL].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()#時間欄位裡的Tab、連續空格都換成單一空格
ev["nsop_time"] = pd.to_datetime(raw, format="%Y/%m/%d %H:%M:%S", errors="coerce")#依2026/10/29 02:22:03的形式轉換時間，轉不了的話變成空值
miss = ev["nsop_time"].isna()
ev.loc[miss, "nsop_time"] = pd.to_datetime(raw[miss], format="mixed", errors="coerce")  # 其他格式備援
n_bad_time = int(ev["nsop_time"].isna().sum())#計算最後仍無法轉換的筆數
ev = (ev.dropna(subset=["nsop_time"]) #刪除時間空值
        .drop_duplicates(["eqp", "nsop_time"]) #刪掉同機台同一秒的重複事件
        .sort_values(["eqp", "nsop_time"]) #依機台與時間排序
        .reset_index(drop=True))
ev["nsop_time"] = ev["nsop_time"].astype("datetime64[ns]")#時間統一成奈秒精度。RPM那邊也是這個精度，兩邊一致這樣後面才可以配對
ev["event_id"] = np.arange(len(ev))#每個事件編號0,1,2....，方便追蹤

# ---------- RPM 檔案依機台分組 ----------
pq_files = sorted(glob.glob(os.path.join(PARQUET_DIR, "*.parquet")))
files_by_eqp = {}
for f in pq_files:#從parquet檔名取出機台編號並建立成一個字典，之後如果要讀取特定機台，只需要讀他自己的parquet檔案
    files_by_eqp.setdefault(os.path.basename(f).split("__")[0].strip().upper(), []).append(f)

MAP_COLS = [RPM_TIME_COL, "Lot Id", "Recipe"]

def load_rpm(eqp, cols):
    """讀一台機台的 RPM 資料（只讀指定欄位），依時間排序。"""
    parts = []
    for f in files_by_eqp.get(eqp, []):#只取出該機台的parquet檔案
        use = [c for c in cols if c in pq.read_schema(f).names]#pq.read_schema:只讀檔案的欄位清單，不讀資料，用來確認欄位存在，避免某個檔案缺欄位時報錯
        parts.append(pd.read_parquet(f, columns=use))
    if not parts:
        return None
    d = pd.concat(parts, ignore_index=True)#將這些parquet檔案合併成一個dataframe
    d["rpm_time"] = pd.to_datetime(d[RPM_TIME_COL], errors="coerce").astype("datetime64[ns]")
    return d.dropna(subset=["rpm_time"]).sort_values("rpm_time", kind="stable").reset_index(drop=True)


# ---------- 用「機台 + 時間」對應 ----------
maps, typical_gaps = [], []
for eqp, g in ev.groupby("eqp"):#先從ev資料(從csv資料來的)中用機台編號來分組，eqp是機台編號、g是每筆分組後的資料
    g = g.sort_values("nsop_time")#再依時間排序
    rpm = load_rpm(eqp, MAP_COLS)#讀取該機台資料(rpm資料)
    if rpm is None: #如果出現在csv檔案(錯誤事件資料)的機台，但卻在rpm資料裡沒有的機台#直接保留這些事件，並標記has_rpm=False
        maps.append(g.assign(has_rpm=False))
        continue

    # 事件時間最接近的那一筆 RPM 紀錄：當時在打哪個 Lot / 導線架 / IC / 線
    m = pd.merge_asof(g, rpm.drop(columns=[RPM_TIME_COL]),#先拿掉原本在rpm原始資料裡的"Timestamp"欄位，因為之後不會用到了。mapping時是使用rpm_time這個經過整理後的時間
                      left_on="nsop_time", right_on="rpm_time", direction="nearest",
                      tolerance=pd.Timedelta(minutes=MATCH_TOL_MIN))
    m["has_rpm"] = True

    # pd.merge_asof: 
    # •	對每個 NSOP 事件，在 RPM 裡找時間最接近的一筆（direction="nearest"，前後都算）。
	# •	超過 MATCH_TOL_MIN（10 分鐘）就不配，那幾個欄位留空。
	# •	配到後，把那一筆的 Lot、Recipe、Wire Id 等欄位接到事件旁邊。

    # 事件前最後一筆、事件後第一筆 RPM 紀錄：NSOP 會停機，事件後應該有一段明顯的空白
    t = rpm["rpm_time"].values
    et = m["nsop_time"].values.astype("datetime64[ns]")
    i = np.searchsorted(t, et, side="right")#用 np.searchsorted 找出事件時間在 RPM 時間軸上的位置
    m["prev_row_time"] = np.where(i > 0, t[np.maximum(i - 1, 0)], np.datetime64("NaT", "ns"))
    m["next_row_time"] = np.where(i < len(t), t[np.minimum(i, len(t) - 1)], np.datetime64("NaT", "ns"))
    if len(t) > 1:
        typical_gaps.append(np.median(np.diff(t)) / np.timedelta64(1, "s"))#這台機台正常情況下，兩筆紀錄間隔多久:把排序後的時間兩兩相減，取中位數，用來和事件前後的間隔比較

    maps.append(m)
    del rpm#處理完就刪掉這台機台的資料，釋放記憶體
    gc.collect()

df_map = pd.concat(maps, ignore_index=True)
df_map["diff_to_rpm_sec"] = (df_map["rpm_time"] - df_map["nsop_time"]).dt.total_seconds()#最接近的那筆RPM和NSOP發生事件差幾秒
df_map["sec_from_prev_row"] = (df_map["nsop_time"] - df_map["prev_row_time"]).dt.total_seconds()#事件前最後一筆紀錄，距離事件幾秒
df_map["sec_to_next_row"] = (df_map["next_row_time"] - df_map["nsop_time"]).dt.total_seconds()#事件後過幾秒才有下一筆紀錄


# ---------- 檢查對應得對不對：Lot / Program / Device 是否一致 ----------
def same(a, b):
    return (df_map[a].astype(str).str.strip().str.upper()
            == df_map[b].astype(str).str.strip().str.upper())

checks = [("LOTNO", "Lot Id", "Lot"), ("PROGRAMNAME", "Recipe", "Program / Recipe")]

print(f"NSOP 事件數：{len(ev):,}（時間無法解析而排除：{n_bad_time}）")
print(f"機台有 RPM 資料的事件：{int(df_map['has_rpm'].sum()):,}")
matched = df_map["rpm_time"].notna()
print(f"{MATCH_TOL_MIN} 分鐘內找得到 RPM 紀錄的事件：{int(matched.sum()):,}")

print("\n【對應品質 1】事件當下的 Lot / Program / Device 是否和 RPM 一致")
for a, b, label in checks:
    if a in df_map.columns and b in df_map.columns:
        both = matched & df_map[a].notna() & df_map[b].notna()
        eq = both & same(a, b)
        print(f"  {label}：兩邊都有值 {int(both.sum())} 筆，一致 {int(eq.sum())} 筆")
        df_map[f"{label}_match"] = np.where(both, eq, np.nan)

print("\n【對應品質 2】NSOP 停機：事件前後 RPM 紀錄的間隔（秒）")
if typical_gaps:
    print(f"  正常情況下相鄰兩筆紀錄間隔的中位數：約 {np.median(typical_gaps):.2f} 秒")
sub = df_map.loc[df_map["has_rpm"] == True, ["sec_from_prev_row", "sec_to_next_row"]]
print(sub.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

display(df_map.head())

NSOP 事件數：186（時間無法解析而排除：0）
機台有 RPM 資料的事件：179
5 分鐘內找得到 RPM 紀錄的事件：109

【對應品質 1】事件當下的 Lot / Program / Device 是否和 RPM 一致
  Lot：兩邊都有值 109 筆，一致 109 筆
  Program / Recipe：兩邊都有值 99 筆，一致 98 筆

【對應品質 2】NSOP 停機：事件前後 RPM 紀錄的間隔（秒）
  正常情況下相鄰兩筆紀錄間隔的中位數：約 0.14 秒
       sec_from_prev_row  sec_to_next_row
count              166.0            126.0
mean             38296.7           6528.2
std              86762.3          23479.4
min                  0.0              0.0
10%                  0.1              0.0
25%                  0.2              0.1
50%                  2.6            151.8
75%              37257.8           1100.4
90%             107980.6          10428.6
max             513927.0         177346.6


,MACHINECODE,EVENTTIME,LOTNO,PROGRAMNAME,DEVICE,eqp,nsop_time,event_id,Lot Id,Recipe,rpm_time,has_rpm,prev_row_time,next_row_time,diff_to_rpm_sec,sec_from_prev_row,sec_to_next_row,Lot_match,Program / Recipe_match
0,WBR0001,2026/11/03 14:00:05,M26BZ8552.00,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-MSTK-SS-BA-HVM-CIP,WBR0001,2026-11-03 14:00:05,0,M26BZ8552.00,SMIM97AGG328-00-MP-00,2026-11-03 14:00:04.933,True,2026-11-03 14:00:04.933,2026-11-03 14:00:05.083,-0.067,0.067,0.083,1.0,1.0
1,WBR0001,2026/11/03 14:21:31,M26BZ8552.00,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-MSTK-SS-BA-HVM-CIP,WBR0001,2026-11-03 14:21:31,1,M26BZ8552.00,SMIM97AGG328-00-MP-00,2026-11-03 14:21:30.900,True,2026-11-03 14:21:30.900,2026-11-03 14:21:31.150,-0.100,0.100,0.150,1.0,1.0
2,WBR0001,2026/11/04 23:37:43,M26BZ4687.00,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,WBR0001,2026-11-04 23:37:43,2,M26BZ4687.00,SMIM97AGG328-00-MP-00,2026-11-04 23:37:40.016,True,2026-11-04 23:37:40.016,2026-11-04 23:38:09.166,-2.984,2.984,26.166,1.0,1.0
3,WBR0001,2026/11/05 10:49:26,M26BZ4492.00,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,WBR0001,2026-11-05 10:49:26,3,M26BZ4492.00,SMIM97AGG328-00-MP-00,2026-11-05 10:49:24.800,True,2026-11-05 10:49:24.800,2026-11-05 10:58:00.700,-1.200,1.200,514.700,1.0,1.0
4,WBR0001,2026/11/06 22:41:41,M26BY8028.00,SMIM97AGG328-00-MP-00,UM2377XLPTIP30996-STK-SS-BA-HVM-CIP,WBR0001,2026-11-06 22:41:41,4,M26BY8028.00,SMIM97AGG328-00-MP-00,2026-11-06 22:42:29.483,True,2026-11-06 22:20:03.350,2026-11-06 22:42:29.483,48.483,1297.650,48.483,1.0,1.0


In [12]:
# ================= 替每一筆 RPM 紀錄加上「距離 NSOP 多遠」的欄位（含事件群處理）=================
# GOOD_SEC = 60            # 事件「之前」幾秒內要有 RPM 紀錄，才拿來當預測目標
EPISODE_GAP_SEC = 1800   # 同機台兩次 NSOP 相隔在這個秒數內，視為同一個事件群#30分鐘
DROP_COLS = ["Errors"]   # 不要帶進標記檔的欄位

os.makedirs(LABEL_DIR, exist_ok=True)

evt = df_map[df_map["has_rpm"] == True].sort_values(["eqp", "nsop_time"]).copy()
# ---------- 每台機台的 RPM 時間軸（只讀一次，後面重複使用）----------
rpm_times = {}
for eqp in evt["eqp"].unique():
    rpm_times[eqp] = load_rpm(eqp, [RPM_TIME_COL])["rpm_time"].values

# ---------- 排除不在 RPM 資料範圍內的事件 ----------
cov = pd.DataFrame([{"eqp": e, "rpm_start": t[0], "rpm_end": t[-1], "rows": len(t)}
                    for e, t in rpm_times.items()])
evt = evt.merge(cov[["eqp", "rpm_start", "rpm_end"]], on="eqp", how="left")
in_cov = (evt["nsop_time"] >= evt["rpm_start"]) & (evt["nsop_time"] <= evt["rpm_end"])
print(f"不在 RPM 資料範圍內而排除的事件：{int((~in_cov).sum())} 個")
evt = evt[in_cov].sort_values(["eqp", "nsop_time"]).reset_index(drop=True)

# ---------- 合併成事件群 ----------
gap = evt.groupby("eqp")["nsop_time"].diff().dt.total_seconds()
evt["episode_id"] = (gap.isna() | (gap > EPISODE_GAP_SEC)).cumsum()
evt["is_first"] = ~evt.duplicated("episode_id")#duplicated("episode_id")對同一編號第二次以後出現的列回傳True，前面加~反轉，就變成只有第一次是True

MIN_ROWS_BEFORE = 500     # 事件前 LOOKBACK_SEC 秒內，至少要有這麼多筆 RPM 紀錄事件前三十分鐘內，至少要500筆rpm紀錄，才能當預測目標
LOOKBACK_SEC = 1800

def rows_before(eqp, when): #此函數用於計算某機台在某個時間點之前30分鐘內有幾筆記錄
    if eqp not in rpm_times:
        rpm_times[eqp] = load_rpm(eqp, [RPM_TIME_COL])["rpm_time"].values
    t = rpm_times[eqp]
    lo = np.searchsorted(t, np.datetime64(when - pd.Timedelta(seconds=LOOKBACK_SEC), "ns"))
    hi = np.searchsorted(t, np.datetime64(when, "ns"), side="right")
    return hi - lo

evt["rows_before"] = [rows_before(e, w) for e, w in zip(evt["eqp"], evt["nsop_time"])]#對每個事件都計算一次row_before
evt["is_target"] = evt["is_first"] & (evt["rows_before"] >= MIN_ROWS_BEFORE)#可當預測目標的事件必須同時滿足兩個條件:是事件群的第一件(is_first)，而且事件前有足夠的資料(row_before>=500)
del rpm_times #刪掉暫存的時間軸，釋放記憶體
gc.collect()

display(evt[["eqp", "nsop_time", "episode_id", "is_first", "rows_before", "is_target"]])

print(f"RPM 範圍內的 NSOP 事件：{len(evt)}")
print(f"合併後的事件群：{evt['episode_id'].nunique()}")
print(f"可當預測目標的事件群：{int(evt['is_target'].sum())}")

summary = []
for eqp, files in files_by_eqp.items():#逐台機台準備
    e = evt[evt["eqp"] == eqp]
    et_all = e["nsop_time"].values.astype("datetime64[ns]")                       # 這台機台所有NSOP的時間
    ep_all = e["episode_id"].values                                               #每次NSOP屬於哪個事件群
    et_tgt = e.loc[e["is_target"], "nsop_time"].values.astype("datetime64[ns]")   # 每個事件群的第一次

    # 先只讀時間欄位，建立整台機台的時間軸（用來算距離事件還有幾筆紀錄）
    all_t = np.sort(np.concatenate([
        pd.to_datetime(pd.read_parquet(f, columns=[RPM_TIME_COL])[RPM_TIME_COL], errors="coerce")
          .dropna().values.astype("datetime64[ns]") for f in files]))
    pos_all = np.searchsorted(all_t, et_all, side="right")     #到NSOP發生為止，計算總共有幾筆RPM紀錄
    pos_tgt = np.searchsorted(all_t, et_tgt, side="right")     #只對可當預測目標的NSOP(每個族群的第一次)，計算總共有幾筆RPM紀錄

    n_rows = 0
    for f in files: #逐個檔案處理
        df_f = pd.read_parquet(f)
        df_f = df_f.drop(columns=[c for c in DROP_COLS if c in df_f.columns]) #拿掉Error欄位
        df_f[RPM_TIME_COL] = pd.to_datetime(df_f[RPM_TIME_COL], errors="coerce").astype("datetime64[ns]")#將Timestamp從文字格是轉成時間格式
        df_f = df_f.dropna(subset=[RPM_TIME_COL]).sort_values(RPM_TIME_COL, kind="stable").reset_index(drop=True) #依時間排序
        t = df_f[RPM_TIME_COL].values  
        pos = np.searchsorted(all_t, t, side="right") #到這筆紀錄為止(含自己)，累積了幾筆紀錄
        n = len(t)

        # 距離「下一個事件群的第一次 NSOP」
        sec_next, rows_next = np.full(n, np.nan), np.full(n, np.nan)
        if len(et_tgt):
            nxt = np.searchsorted(et_tgt, t, side="left") #找出每筆紀錄之後的第一個目標事件
            ok = nxt < len(et_tgt) #標出後面還有目標事件的紀錄，如果最後一個目標事件之後的資料是False
            ni = np.minimum(nxt, len(et_tgt) - 1)#避免索引超出範圍
            sec_next = np.where(ok, (et_tgt[ni] - t) / np.timedelta64(1, "s"), np.nan)#時間差，換算成秒
            rows_next = np.where(ok, pos_tgt[ni] - pos, np.nan)#位置相減，就是中間還有幾筆紀錄

        # 距離「上一次 NSOP」（任何一次），以及是否夾在同一個事件群中間
        sec_prev, rows_prev = np.full(n, np.nan), np.full(n, np.nan)
        in_episode = np.zeros(n, dtype=bool)
        if len(et_all):
            nxa = np.searchsorted(et_all, t, side="left")#找出每筆紀錄之後的第一次NSOP
            prv = nxa - 1
            okp = prv >= 0
            pi = np.maximum(prv, 0)
            sec_prev = np.where(okp, (t - et_all[pi]) / np.timedelta64(1, "s"), np.nan)
            rows_prev = np.where(okp, pos - pos_all[pi], np.nan)
            okn = nxa < len(et_all)
            in_episode = okp & okn & (ep_all[pi] == ep_all[np.minimum(nxa, len(et_all) - 1)])

        df_f["sec_to_next_nsop"] = sec_next  #距離下一次目標事件還有幾秒
        df_f["rows_to_next_nsop"] = rows_next  #距離下一個目標事件還有幾筆紀錄
        df_f["sec_since_prev_nsop"] = sec_prev #距離上一次NSOP(任何一次)過了幾秒
        df_f["rows_since_prev_nsop"] = rows_prev#距離上一次NSOP(任何一次)過了幾筆
        df_f["in_nsop_episode"] = in_episode#是否夾在同一個事件群的兩次NSOP之間

        df_f.to_parquet(os.path.join(LABEL_DIR, os.path.basename(f)), index=False)
        n_rows += n
        del df_f
        gc.collect()

    summary.append({"eqp": eqp, "RPM筆數": n_rows, "NSOP事件": len(et_all),
                    "事件群": int(e["episode_id"].nunique()), "可當目標的事件群": len(et_tgt)})

df_label_summary = pd.DataFrame(summary)
print(f"完成，存在：{LABEL_DIR}")
display(df_label_summary)

不在 RPM 資料範圍內而排除的事件：66 個


,eqp,nsop_time,episode_id,is_first,rows_before,is_target
0,WBR0001,2026-11-03 14:00:05,1,True,7154,True
1,WBR0001,2026-11-03 14:21:31,1,False,4005,False
2,WBR0001,2026-11-04 23:37:43,2,True,730,True
3,WBR0001,2026-11-05 10:49:26,3,True,8996,True
4,WBR0001,2026-11-06 22:41:41,4,True,2540,True
...,...,...,...,...,...,...
108,WBR1040,2026-11-02 22:25:03,88,False,3789,False
109,WBR1040,2026-11-02 22:27:52,88,False,3125,False
110,WBR1040,2026-11-03 15:54:19,89,True,3841,True
111,WBR1040,2026-11-04 22:24:42,90,True,0,False


RPM 範圍內的 NSOP 事件：113
合併後的事件群：91
可當預測目標的事件群：84
完成，存在：C:\Users\A134549\rpm_parquet_labeled_11021108


,eqp,RPM筆數,NSOP事件,事件群,可當目標的事件群
0,WBR0001,1796393,5,4,4
1,WBR0004,2405145,2,2,1
2,WBR0005,1959618,9,4,3
3,WBR0038,79908,0,0,0
4,WBR1001,1786431,2,2,2
5,WBR1004,2343530,2,2,2
6,WBR1006,3290,0,0,0
7,WBR1008,615417,0,0,0
8,WBR1014,1864889,2,2,2
9,WBR1018,1034666,13,12,12


在113個事件哩，有91個事件群，說明有22次NSOP是連續發生的後續事件。84個可當目標，說明有7個事件群在發生前沒有足夠的。事件群判斷規則:同一台機台的NSOP依時間排序，如果和前一次NSOP的間隔不超過30分鐘，就算同一群:超過30分鐘就開始新的一群。  

In [13]:
# ================= 產生 label（可以反覆改參數重跑，不必重做上一個 cell）=================
WINDOW_ROWS = 2000    # NSOP 前幾筆紀錄算「異常」
WINDOW_SEC = 1800     # 而且必須在 NSOP 前幾秒以內（避免跨過 RPM 空窗）
BUFFER_ROWS = 20      # 緊貼事件的最後幾筆不用（事件時間只到秒，有誤差）
RECOVER_ROWS = 500    # NSOP 之後的恢復期不用

use_cols = ["Timestamp", "EQP", "Lot Id", "Leadframe Id", "Device Id", "Wire Id", "Bond Type",
            "USG current (mA)", "Force", "EFO Voltage Avg (V)",
            "sec_to_next_nsop", "rows_to_next_nsop", "sec_since_prev_nsop", "rows_since_prev_nsop","in_nsop_episode"]
first_file = sorted(glob.glob(os.path.join(LABEL_DIR, "*.parquet")))[0]
use_cols = [c for c in use_cols if c in pq.read_schema(first_file).names]   # 只留檔案裡真的有的欄位
df_l = read_dir(LABEL_DIR, use_cols)

df_l["label"] = ((df_l["rows_to_next_nsop"] <= WINDOW_ROWS) &
                 (df_l["sec_to_next_nsop"] <= WINDOW_SEC)).astype("int8")
exclude = ((df_l["rows_to_next_nsop"] <= BUFFER_ROWS) |
           (df_l["rows_since_prev_nsop"] <= RECOVER_ROWS) |
           df_l["in_nsop_episode"])
df_l = df_l[~exclude].reset_index(drop=True)

print(f"可用資料：{len(df_l):,} 筆")
print(f"label = 1（NSOP 前的異常窗）：{int(df_l['label'].sum()):,} 筆，佔 {df_l['label'].mean():.3%}")
display(df_l.groupby("EQP")["label"].agg(筆數="size", 異常筆數="sum"))

可用資料：26,775,795 筆
label = 1（NSOP 前的異常窗）：160,904 筆，佔 0.601%


,筆數,異常筆數
EQP,,
WBR0001,1791216,6649
WBR0004,2404124,1980
WBR0005,1957554,5940
WBR0038,79908,0
WBR1001,1785389,3960
WBR1004,2342488,3960
WBR1006,3290,0
WBR1008,615417,0
WBR1014,1863847,3960


In [14]:
# df_wbr=df[df['EQP']=='WBR0004']
# df_wbr=df_wbr.sort_values(['Timestamp'])
# df_wbr[(df_wbr["Timestamp"].between("2026-10-27 09:30:56", "2026-10-27 10:15:56"))]
# df_wbr